IMPORTING MODULES

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import pyedflib
from mne import create_info
from mne.io import RawArray
from mne.time_frequency import psd_array_welch
from antropy import hjorth_params
from antropy.entropy import spectral_entropy, perm_entropy
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import math
import torch.optim as optim

PREPROCESSING AND FEATURE EXTRACTION

In [ ]:
def load_edf_raw(file_path):
    f = pyedflib.EdfReader(file_path)
    n_channels = f.signals_in_file
    all_data = [f.readSignal(i) for i in range(n_channels)]
    ch_names = [f.getLabel(i).strip() for i in range(n_channels)]
    sfreq = f.getSampleFrequency(0)
    f.close()

    data = np.array(all_data)
    info = create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    raw = RawArray(data, info)
    return raw

In [ ]:
def preprocess_edf(file_path, window_sec=5):
    try:
        raw = load_edf_raw(file_path)
        data, _ = raw.get_data(return_times=True)
        sfreq = raw.info['sfreq']
        n_channels, n_samples = data.shape

        window_size = int(window_sec * sfreq)
        n_windows = n_samples // window_size
        if n_windows == 0:
            return None

        bands = [(0.5, 4), (4, 8), (8, 12), (12, 16), (16, 25), (25, 30), (30, 40)]
        band_indices = None
        features = []

        for w in range(n_windows):
            start = w * window_size
            stop = start + window_size
            window_data = data[:, start:stop]

            psd, freqs = psd_array_welch(window_data, sfreq=sfreq, fmin=0.5, fmax=40, n_fft=256, n_jobs=1)
            if band_indices is None:
                band_indices = [np.logical_and(freqs >= fmin, freqs <= fmax) for fmin, fmax in bands]

            psd_bands = np.stack([psd[:, idx].mean(axis=1) for idx in band_indices], axis=1)
            window_features = []

            for ch in range(n_channels):
                x = window_data[ch]
                ch_psd = psd_bands[ch].tolist()
                hjorth = hjorth_params(x)
                spec_ent = spectral_entropy(x, sfreq, method="welch", normalize=True)
                perm_ent = perm_entropy(x, normalize=True)

                window_features.extend(ch_psd + list(hjorth) + [spec_ent, perm_ent])

            features.append(window_features)

        return np.array(features)

    except Exception as e:

        return None



DATASET

In [ ]:
root_folder = "HUP_IEEG"
folders = [("failure", 0), ("success", 1)]


In [ ]:
all_X, all_y = [], []
skipped_files = 0

for folder_name, label in folders:
    folder_path = os.path.join(root_folder, folder_name)
    count = 0

    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            if f.lower().endswith(".edf"):
                file_path = os.path.join(dirpath, f)
                X_trial = preprocess_edf(file_path, window_sec=5)
                if X_trial is not None and X_trial.shape[0] > 0:
                    all_X.append(X_trial)
                    all_y.append(label)
                    count += 1
                else:
                    skipped_files += 1

                if count >= 200:
                    break
        if count >= 200:
            break

In [ ]:

if all_X:
    cleaned_X, cleaned_y = [], []
    for x, label in zip(all_X, all_y):
        if x is not None and x.shape[0] > 0:
            cleaned_X.append(x)
            cleaned_y.append(label)

    if not cleaned_X:
        raise ValueError("No valid trials")

    max_windows = max(x.shape[0] for x in cleaned_X)
    max_features = max(x.shape[1] for x in cleaned_X)

    X_final = np.zeros((len(cleaned_X), max_windows, max_features))
    valid_lengths = []

    for i, x in enumerate(cleaned_X):
        n_win, n_feat = x.shape
        X_final[i, :n_win, :n_feat] = x
        valid_lengths.append(n_win)

    y = np.array(cleaned_y)

    mean = X_final.mean(axis=(0, 1), keepdims=True)
    std = X_final.std(axis=(0, 1), keepdims=True) + 1e-8
    X = (X_final - mean) / std
    X = np.nan_to_num(X, nan=0.0)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    X = X[indices]
    y = y[indices]
    valid_lengths = np.array(valid_lengths)[indices]
    np.save("EEG_features_surgery.npy", X)
    np.save("EEG_labels_surgery.npy", y)
    np.save("EEG_lengths_surgery.npy", valid_lengths)

LOADING INPUT

In [ ]:
X = np.load("EEG_features_surgery.npy")
y = np.load("EEG_labels_surgery.npy")

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)


SPLITING DATASET

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

TRANSFORMER MODEL WITH TIME2VEC

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, hidden, drop_prob=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, hidden)
        self.linear2 = nn.Linear(hidden, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=drop_prob)

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [ ]:
class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, q, k, v, mask=None):
        _, _, _, d_tensor = k.size()
        k_t = k.transpose(2, 3)
        score = (q @ k_t) / math.sqrt(d_tensor)
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e9)
        score = self.softmax(score)
        return score @ v, score

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.attention = ScaleDotProductAttention()
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_concat = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)
        q, k, v = self.split(q), self.split(k), self.split(v)
        out, _ = self.attention(q, k, v, mask=mask)
        return self.w_concat(self.concat(out))

    def split(self, tensor):
        batch_size, length, d_model = tensor.size()
        d_tensor = d_model // self.n_head
        return tensor.view(batch_size, length, self.n_head, d_tensor).transpose(1, 2)

    def concat(self, tensor):
        batch_size, head, length, d_tensor = tensor.size()
        return tensor.transpose(1, 2).contiguous().view(batch_size, length, head * d_tensor)


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-12):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, unbiased=False, keepdim=True)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

In [ ]:
class Time2Vec(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.d_model = d_model
        self.w0 = nn.Parameter(torch.randn(1, 1))
        self.b0 = nn.Parameter(torch.randn(1, 1))
        self.w = nn.Parameter(torch.randn(1, d_model - 1))
        self.b = nn.Parameter(torch.randn(1, d_model - 1))

    def forward(self, t):
        linear_term = self.w0 * t + self.b0
        periodic_terms = torch.sin(self.w * t + self.b)
        return torch.cat([linear_term, periodic_terms], dim=-1)

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, hidden_dim, drop_prob=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_head)
        self.norm1 = LayerNorm(d_model)
        self.ffn = PositionwiseFeedForward(d_model, hidden_dim, drop_prob)
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(drop_prob)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attention(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_head, hidden_dim, num_layers, drop_prob=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_head, hidden_dim, drop_prob)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [ ]:
class EEGTransformer(nn.Module):
    def __init__(self, feature_dim=228, d_model=128, n_head=8,
                 hidden_dim=512, num_layers=4, drop_prob=0.5):
        super().__init__()
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.time2vec = Time2Vec(d_model=d_model)
        self.encoder = TransformerEncoder(d_model, n_head, hidden_dim, num_layers, drop_prob)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x, mask=None, predict=False):
        batch_size, seq_len, _ = x.shape
        x_proj = self.input_proj(x)
        t = torch.arange(seq_len, device=x.device).unsqueeze(0).unsqueeze(-1).repeat(batch_size, 1, 1).float()
        t2v = self.time2vec(t)
        x = x_proj + t2v
        x = self.encoder(x, mask)
        x = x.mean(dim=1)
        logits = self.classifier(x)
        if predict:
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            return preds
        return logits

In [ ]:
class EEGTransformer(nn.Module):
    def __init__(self, feature_dim=228, d_model=128, n_head=8,
                 hidden_dim=512, num_layers=4, drop_prob=0.5):
        super().__init__()
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.time2vec = Time2Vec(d_model=d_model)
        self.encoder = TransformerEncoder(d_model, n_head, hidden_dim, num_layers, drop_prob)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x, mask=None, predict=False):
        batch_size, seq_len, _ = x.shape
        x_proj = self.input_proj(x)
        t = torch.arange(seq_len, device=x.device).unsqueeze(0).unsqueeze(-1).repeat(batch_size, 1, 1).float()
        t2v = self.time2vec(t)
        x = x_proj + t2v
        x = self.encoder(x, mask)
        x = x.mean(dim=1)
        logits = self.classifier(x)
        if predict:
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            return preds
        return logits

TRAINING AND EVALUATING MODEL

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(" Using device:", device)

feature_dim = X.shape[2]
model = EEGTransformer(feature_dim=feature_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc = 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        preds = (torch.sigmoid(logits) >= 0.5).long()
        acc = (preds == y_batch.long()).float().mean()
        train_loss += loss.item()
        train_acc += acc.item()

    model.eval()
    val_loss, val_acc = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            acc = (preds == y_batch.long()).float().mean()
            val_loss += loss.item()
            val_acc += acc.item()

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}, Acc: {val_acc/len(val_loader):.4f}")

SAVING MODEL

In [ ]:
torch.save(model.state_dict(), "surgery.h5")